# 02 — Train the plain RNN baseline

**Viraat Chaudhary · DATA 266 Lab 1 · Team 15 · Yelp Polarity**

Use **Colab A100**. Notebook 01 has already completed; do not rerun it. This notebook restores the approved processed ZIP from your existing Drive backup. No dataset download or vocabulary fitting is performed. All weights start randomly; checkpoint selection and early stopping use validation only.

Review the proposed numeric settings and write your own design reasons before training. The brief reserves core decisions and analysis to the student. The RNN/LSTM share embedding size, hidden size, pooling, head and training defaults; their recurrent cells differ. TCN uses a distinct residual causal convolution stack, not Anshika's parallel TextCNN.

Run top-to-bottom. Save the executed notebook using **File → Download → Download .ipynb**. Use the exact filename provided. Outputs persist in your own Drive run folder. Source code is assistant-assisted; student explanations and analysis must be your own.


## Restore the package and packages

In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import drive, files

# Set these once; keep the same RUN_NAME for notebooks 02–07 and resumed sessions.
DRIVE_BASE_RELATIVE = 'DATA266_Lab1_Task2/Viraat_Chaudhary'
RUN_NAME = 'task2_models_rnn_lstm_tcn_v1'
PACKAGE_ZIP_NAME = 'Viraat_Task2_Run_Package.zip'
PACKAGE_ID = 'viraat_yelp_rnn_lstm_tcn_v1'

drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive') / DRIVE_BASE_RELATIVE
ROOT = DRIVE_BASE / RUN_NAME / 'task2_sentiment/Viraat_Chaudhary'
CACHE_ROOT = Path('/content/viraat_task2_data')
if not ROOT.exists():
    archive = DRIVE_BASE / PACKAGE_ZIP_NAME
    if not archive.exists():
        print('Upload Viraat_Task2_Run_Package.zip (the code package, not a dataset ZIP).')
        uploaded = files.upload()
        if PACKAGE_ZIP_NAME not in uploaded:
            raise FileNotFoundError('Select the exact package ZIP filename.')
        archive = Path('/content') / PACKAGE_ZIP_NAME
        archive.write_bytes(uploaded[PACKAGE_ZIP_NAME])
    extraction = Path('/content/viraat_task2_package_extract')
    extraction.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            if not (extraction / info.filename).resolve().is_relative_to(extraction.resolve()):
                raise ValueError('Invalid package ZIP path.')
        if bundle.testzip() is not None:
            raise ValueError('Package ZIP CRC failed.')
        bundle.extractall(extraction)
    manifest = json.loads((extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_text())
    if manifest['package_id'] != PACKAGE_ID:
        raise ValueError('Incorrect package identity.')
    for item in manifest['files']:
        path = extraction / item['path']
        if hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
            raise ValueError('Package integrity failed: ' + item['path'])
    source = extraction / 'task2_sentiment/Viraat_Chaudhary'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, ROOT)
    (ROOT / 'package_installation.json').write_text(json.dumps({'package_id': PACKAGE_ID,
                                                              'package_manifest_sha256': hashlib.sha256(
        (extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_bytes()).hexdigest()}, indent=2) + '\n')
else:
    installed = json.loads((ROOT / 'package_installation.json').read_text())
    if installed['package_id'] != PACKAGE_ID:
        raise ValueError('This RUN_NAME belongs to a different package.')

required = {'numpy': 'numpy==2.1.3', 'scipy': 'scipy>=1.14,<2',
            'sklearn': 'scikit-learn==1.6.1', 'nltk': 'nltk==3.9.1',
            'datasets': 'datasets==4.8.5', 'matplotlib': 'matplotlib==3.10.0',
            'pandas': 'pandas>=2.2,<4', 'tqdm': 'tqdm==4.67.3'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Colab runtime with PyTorch installed.')
# Keep the tokenizer version consistent with the approved snapshot.
import importlib.metadata
if importlib.metadata.version('nltk') != '3.9.1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1'])

source_dir = str(ROOT / 'src')
sys.path.insert(0, source_dir)
for name in ('runtime_utils', 'yelp_data', 'data_runtime', 'models', 'metrics',
             'training', 'evaluation', 'review', 'reporting', 'evaluator'):
    module = sys.modules.get(name)
    if module is not None and Path(getattr(module, '__file__', '')).parent != ROOT / 'src':
        del sys.modules[name]
from data_runtime import verify_preprocessing
verify_preprocessing(ROOT)
import torch
import pandas as pd
print('Member folder:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Approved preprocessing restored; original notebook 01 is preserved.')


Mounted at /content/drive
Upload Viraat_Task2_Run_Package.zip (the code package, not a dataset ZIP).


Saving Viraat_Task2_Run_Package.zip to Viraat_Task2_Run_Package.zip
Member folder: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary
GPU: NVIDIA A100-SXM4-40GB
Approved preprocessing restored; original notebook 01 is preserved.


## Review settings and enter your own explanations

In [2]:
from runtime_utils import read_json, write_json, collect_design_notes
from pprint import pprint

# Optional overrides: change only before any training starts. These are starting values,
# not a claim that they are optimal. Review the three printed configurations.
OVERRIDES = {'baseline': {}, 'experiment_1': {}, 'experiment_2': {}}
if any(OVERRIDES.values()):
    started = any((ROOT / f'outputs/metrics/{n}_training_manifest.json').exists()
                  for n in OVERRIDES)
    if started:
        raise RuntimeError('Settings are already in use. Do not change this run; especially never retune after test.')
    for name, changes in OVERRIDES.items():
        configuration = read_json(ROOT / f'configs/{name}.json')
        configuration.update(changes)
        write_json(ROOT / f'configs/{name}.json', configuration)
for name in OVERRIDES:
    print('\n' + name)
    pprint(read_json(ROOT / f'configs/{name}.json'))

# Enter explanations in your own words. Six prompts appear only the first time.
# Existing explanations are reused when reconnecting/resuming.
DESIGN_NOTES = collect_design_notes(ROOT)
print('Student design explanations saved.')



baseline
{'amp': True,
 'architecture': 'rnn',
 'batch_size': 512,
 'dropout': 0.25,
 'early_stopping_patience': 2,
 'embedding_dim': 128,
 'embedding_dropout': 0.1,
 'gradient_clip_norm': 1.0,
 'head_dim': 64,
 'hidden_dim': 128,
 'learning_rate': 0.001,
 'max_epochs': 5,
 'name': 'baseline',
 'pooling': 'masked mean plus masked maximum',
 'pretrained_weights': False,
 'seed': 2662503,
 'selection': 'validation macro-F1, then lower validation BCE',
 'settings_provenance': 'assistant-proposed starting values; student reviews '
                        'before first training',
 'weight_decay': 0.0001}

experiment_1
{'amp': True,
 'architecture': 'lstm',
 'batch_size': 512,
 'dropout': 0.25,
 'early_stopping_patience': 2,
 'embedding_dim': 128,
 'embedding_dropout': 0.1,
 'gradient_clip_norm': 1.0,
 'head_dim': 64,
 'hidden_dim': 128,
 'learning_rate': 0.001,
 'max_epochs': 5,
 'name': 'experiment_1',
 'pooling': 'masked mean plus masked maximum',
 'pretrained_weights': False,
 'seed': 2

## Restore and verify the approved development arrays

In [3]:
from data_runtime import restore_development
DEV, PROCESSED_DIR = restore_development(ROOT, DRIVE_BASE, CACHE_ROOT)
print('Train:', len(DEV['train']['labels']), '| validation:', len(DEV['validation']['labels']))


Train: 504000 | validation: 56000


## Train or resume

An epoch-boundary checkpoint supports recovery after a disconnection. Completed runs are reused. Interrupted partial epochs are restarted; their raw records are retained and the timing scope is documented.

In [4]:
from training import train_model
TRAINING_RESULT = train_model(ROOT, 'baseline', DEV)
pd.DataFrame([TRAINING_RESULT]).T


baseline epoch 1:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 1: validation macro-F1=0.92650; training phase=10.2s; best epoch=1.


baseline epoch 2:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 2: validation macro-F1=0.93273; training phase=8.3s; best epoch=2.


baseline epoch 3:   0%|          | 0/985 [00:00<?, ?it/s]

...

baseline epoch 4:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 4: validation macro-F1=0.93887; training phase=8.4s; best epoch=4.


baseline epoch 5:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 5: validation macro-F1=0.93700; training phase=8.4s; best epoch=4.


,0
status,completed
model,baseline
identity,968739b9f49a2dcfd4336ad618223e2385057a3bb86287...
config,configs/baseline.json
config_sha256,11ba31a77bea8119b90c54de3e6105b1b886b07ec12522...
data_contract,e3010a4d701170a1bb7e0378fd1049ac391c5031b40452...
implementation_fingerprint,6c9bf154d4817e56ab49065c1fe1a3ab0fa4146f451fe4...
checkpoint,checkpoints/baseline_best.pt
checkpoint_sha256,fa3ae7b0eda530081858a2d471335fef949295bfea924e...
selected_epoch,4


## Save stage evidence

Files are saved in Drive. The small ZIP excludes model weights; the final notebook exports all selected weights. Download this executed notebook before opening the next one.

In [5]:
from reporting import refresh_reports, export_artifacts
refresh_reports(ROOT)
STAGE_ZIP = export_artifacts(ROOT, '02_Training_Evidence')
print('Stage evidence:', STAGE_ZIP)
print('Download the executed notebook with File → Download → Download .ipynb.')


Stage evidence: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary/exports/Viraat_Task2_02_Training_Evidence.zip
Download the executed notebook with File → Download → Download .ipynb.
